# Kubernetes Python Client

In [ ]:
# Day 47 — Kubernetes Python Client
# As a DevOps engineer, you run `kubectl` every day.
# But when building self-healing operators, AIOps auto-scalers, or MLOps
# deployment bots, shelling out to `kubectl` in bash is fragile.
#
# The official `kubernetes` Python client talks directly to the K8s API server!
#
# Today we'll cover:
#   1. Loading kubeconfig vs in-cluster config
#   2. Listing pods, deployments, and filtering by label selector
#   3. Creating & patching deployments programmatically (scaling replicas)
#   4. Watching cluster events in real time (`watch.Watch`)
#   5. Deleting resources safely
#
# Install: pip install kubernetes

### 1. Connecting to Kubernetes & Listing Pods

In [ ]:
# Two ways to authenticate:
#   1. Local dev / CI runner: config.load_kube_config()
#   2. Inside a K8s Pod:      config.load_incluster_config()

list_pods_code = '''from kubernetes import client, config

def get_k8s_clients():
    try:
        config.load_incluster_config()
    except config.ConfigException:
        config.load_kube_config()
    return client.CoreV1Api(), client.AppsV1Api()

v1, apps_v1 = get_k8s_clients()

# List pods in 'default' namespace with a label selector
pods = v1.list_namespaced_pod(namespace="default", label_selector="app=ml-api")
for pod in pods.items:
    restarts = sum(cs.restart_count for cs in (pod.status.container_statuses or []))
    print(f"{pod.metadata.name:30s} | Phase: {pod.status.phase:10s} | Restarts: {restarts}")
'''
print(list_pods_code)

### 2. Creating & Patching Deployments (Scaling Replicas)

In [ ]:
# CoreV1Api handles Pods, Services, ConfigMaps, Secrets.
# AppsV1Api handles Deployments, StatefulSets, DaemonSets.

scale_code = '''from kubernetes import client, config

config.load_kube_config()
apps_v1 = client.AppsV1Api()

def scale_deployment(name: str, namespace: str, replicas: int):
    body = {"spec": {"replicas": replicas}}
    resp = apps_v1.patch_namespaced_deployment_scale(
        name=name,
        namespace=namespace,
        body=body
    )
    print(f"Scaled {name} in {namespace} to {resp.spec.replicas} replicas")

# Example: scale up ML serving API before traffic spike
# scale_deployment("ml-serving-api", "prod", 5)
'''
print(scale_code)

### 3. Watching Cluster Events in Real Time

In [ ]:
# `watch.Watch()` streams live K8s events — great for AIOps alert bots!

watch_code = '''from kubernetes import client, config, watch

config.load_kube_config()
v1 = client.CoreV1Api()
w = watch.Watch()

for event in w.stream(v1.list_namespaced_event, namespace="prod", timeout_seconds=60):
    ev = event["object"]
    if ev.type == "Warning":
        print(f"[WARNING] {ev.involved_object.name}: {ev.reason} -> {ev.message}")
'''
print(watch_code)

In [ ]:
# Practice Questions:
#
# 1. Why should a Python script running inside a K8s pod use `load_incluster_config()`?
# 2. Which API class handles Deployments vs Pods vs HorizontalPodAutoscalers?
# 3. How does `label_selector` improve performance on large clusters?

### DevOps / AI-MLOps Practice

In [ ]:
# Exercise 1: Automated Pod Restarter
# List all pods in a namespace; if any container is in `CrashLoopBackOff`,
# fetch its last 50 log lines (`read_namespaced_pod_log`) and delete the pod.

# Exercise 2: Deployment Scaler
# Write a function `scale_by_prediction(deployment, predicted_rps)` that patches
# `replicas = max(2, math.ceil(predicted_rps / 100))`.

# Exercise 3: Namespace Resource Auditor
# Iterate through all pods in a namespace and flag any container missing
# `resources.limits` or `resources.requests`.